In [ ]:
# Libraries:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Read in PreGame test data and trained models: 
os.chdir("/path/to/PreGame/testing/data/")

meta_validation = pd.read_csv("meta_discovery_tested.tsv", sep='\t')
mtx_validation = pd.read_feather("PreGame_testing_matrix.feather").to_numpy()
models = joblib.load("PreGame_testing_models.pkl")


In [ ]:
# Predict tumor-reactivity using trained models:
K = 100
for k in range(K):
    clf = models[k]

    if k == 0:
        Y_Vali_PROB = clf.predict_proba(mtx_validation)[:,1]
    else:
        Y_Vali_PROB = np.column_stack((Y_Vali_PROB, clf.predict_proba(mtx_validation)[:,1]))

df_cell_pred = pd.concat((meta_validation.loc[:,"Barcode"], pd.DataFrame(np.mean(Y_Vali_PROB, axis=1))), axis=1)
df_cell_pred.columns = ["Cell", "Prediction"]

# Save ffinal 
df_cell_pred.to_csv('PreGame_Predictions.tsv', sep='\t', index=False)